## 5. Analyse
### 5.1 Validation du modèle

In [1]:
import Pkg
Pkg.activate("..")

using JuMP
using HiGHS
include("../src/modele_v3.jl")
include("../src/donnees.jl")

data = charger_donnees()
model, x, e, viol_type, viol_jour, viol_semaine =
    construire_modele(data; p_min=1, p_semaine=1)

optimize!(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 2814 rows; 1300 cols; 6351 nonzeros; 1300 integer variables (1260 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 3e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+01]
Presolving model
406 rows, 288 cols, 1833 nonzeros 0s
278 rows, 284 cols, 1795 nonzeros 0s
274 rows, 284 cols, 1764 nonzeros 0s
Presolve reductions: rows 274(-2540); columns 284(-1016); nonzeros 1764(-4587) 
Objective function is integral with scale 1

Solving MIP model with:
   274 rows
   284 cols (244 binary, 40 integer, 0 implied int., 0 continuous, 0 domain fixed)
   1764 nonzeros
   Thread count 8 (of 16 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L

  Activating project at `C:\Dev\workforce-scheduling-optimization`


In [2]:
assert_is_solved_and_feasible(model)

In [3]:
# Constantes
I = data.I
J = data.J
K = data.K
T = data.T
S = data.S
D = data.D 
N = data.N 
nb_personnes = data.nb_personnes
C = data.C 
S_max = data.S_max 
S_min = data.S_min
TypeShift = data.TypeShift
Jour = data.Jour
Succ= data.Succ
PrefType = data.PrefType
PrefJour= data.PrefJour
PoidsPref = data.PoidsPref
TypeShiftBool = data.TypeShiftBool
JourShiftBool = data.JourShiftBool;

In [4]:
valeur_x = value(x)
valeur_e = value(e)
valeur_type = value(viol_type)
valeur_jour = value(viol_jour)
valeur_semaine = value(viol_semaine)

for i in I
    shifts = [j for j in J if value(x[i,j]) > 0.5]
    println("Employé $i : ", join(shifts, ", "))
end

println("Contraintes violées : ")
for i in I
    if valeur_e[i] > 0 
        println("Employé qui fait moins de shifts que souhaité $i : ",  valeur_e[i])
    end
    for j in J
        if valeur_type[i,j] > 0
            println("Type de shift Employé/Shift $i $j : ", valeur_type[i,j])
        end
        
        if valeur_jour[i,j] > 0
            println("Jour Employé/Shift $i $j : ", valeur_jour[i,j])
        end
    end

    if valeur_semaine[i] > 0
        println("Semaine trop chargée Employé $i : ", valeur_semaine[i])
    end
end


Employé E01 : S04, S07, S10, S13, S19
Employé E02 : S04, S07, S10, S16, S19
Employé E03 : S08, S12, S17, S20
Employé E04 : S05, S08, S11, S14, S17
Employé E05 : S02, S10, S16, S17, S20
Employé E06 : S01, S07, S13, S16, S19
Employé E07 : S02, S05, S08, S11, S14
Employé E08 : S06, S09, S15, S18
Employé E09 : S02, S08, S14, S20
Employé E10 : S01, S04, S09, S13, S21
Employé E11 : S05, S11, S17
Employé E12 : S01, S04, S10, S13
Employé E13 : S01, S04, S07, S13, S16, S19
Employé E14 : S03, S12, S15, S21
Employé E15 : S02, S05, S08, S11, S14
Employé E16 : S01, S04, S07, S10, S13
Employé E17 : S09, S12, S20, S21
Employé E18 : S03, S06, S11, S14, S15, S18
Employé E19 : S03, S06, S15, S18
Employé E20 : S02, S05, S08, S14, S17
Contraintes violées : 
Type de shift Employé/Shift E03 S12 : 1.0
Employé qui fait moins de shifts que souhaité E05 : 1.0
Jour Employé/Shift E05 S02 : 1.0
Jour Employé/Shift E05 S10 : 1.0
Jour Employé/Shift E05 S20 : 1.0
Employé qui fait moins de shifts que souhaité E09 : 2.0

On vérifie si les contraintes dures sont respectées.

In [5]:
begin
	# un employé n'est pas affecté sur un shift où il n'est pas disponible
	cpt = 0
	for (n,i) in enumerate(I)
		for(m,j) in enumerate(J)
			if round(Int,valeur_x[i,j]) == 1 && D[n,m] == 0
				cpt += 1
			end
		end
	end
	print("On a $cpt problèmes de disponibilité.")
end

On a 0 problèmes de disponibilité.

In [6]:
begin
	# problèmes avec le nombre de personnes affectées à chaque shift
	cpt1 = 0
	for (m,j) in enumerate(J)
		if round(Int,sum( valeur_x[i,j] for i in I)) != nb_personnes[m]
			cpt1+=1
		end
	end
	print("On a $cpt1 problèmes avec le nombre de personnes affectées par shift.")
end

On a 0 problèmes avec le nombre de personnes affectées par shift.

In [7]:
begin
	# chaque employé ne dépasse pas son nombre maximal de shifts
	cpt2 = 0
	for (n,i) in enumerate(I)
		if round(Int,sum( valeur_x[i,j] for j in  J) ) > S_max[n]
			cpt2+=1
		end
	end
	print("On a $cpt2 employés qui dépassent leur nombre de shifts maximum.")
end

On a 0 employés qui dépassent leur nombre de shifts maximum.

In [8]:
begin
	# Contrôle des compétences nécessaires par shift
	cpt3=0
	for (n,j) in enumerate(J) #pour chaque shift
		for (m,k) in enumerate(K) #pour chaque compétence
			nb_employes_c = sum(round(Int,valeur_x[I[i],j])*C[i,m] for i in eachindex(I))
			if nb_employes_c < N[n,m] 
				cpt3 +=1
			end
		end
	end
	print("Il manque $cpt3 couples (shifts/compétence).")
end

Il manque 0 couples (shifts/compétence).

$x[i,j]$ étant binaire, un employé occupe une place par shift.

In [9]:
begin
	# les contraintes de repos sont respectées
	cpt5 = 0
	for (n,i) in enumerate(I)
		for (m,j) in enumerate(J)
			if m < length(J)
				if round(Int, valeur_x[i,J[m]]) == 1 && round(Int, valeur_x[i,J[m+1]]) == 1 
					if Succ[TypeShift[J[m]],TypeShift[J[m+1]]] == 0 # succession non autorisée
						cpt5 +=1
	 				end
				end
			end
            if m < length(J) - 1
				if round(Int, valeur_x[i,J[m]]) == 1 && round(Int, valeur_x[i,J[m+2]]) == 1 
					if Succ[TypeShift[J[m]],TypeShift[J[m+2]]] == 0 # succession non autorisée
						cpt5 +=1
	 				end
				end
			end
		end
	end
	print("Il y a $cpt5 contraintes de repos qui ne sont pas respectées.")
end

Il y a 0 contraintes de repos qui ne sont pas respectées.

Toutes les contraintes dures sont respectées. 

Le solveur a prouvé l'optimalité du résultat. On va donc vérifier la cohérence des pénalités avec les violations observées.

### 5.2 Vérification des pénalités

On a trouvé ces informations plus haut :

Contraintes violées (poids = 1 sauf si précisé autrement) : 
* Type de shift Employé/Shift E03 S12 : 1.0 (poids : 3)
* Employé qui fait moins de shifts que souhaité E05 : 1.0
* Jour Employé/Shift E05 S02 : 1.0
* Jour Employé/Shift E05 S10 : 1.0
* Jour Employé/Shift E05 S20 : 1.0
* Employé qui fait moins de shifts que souhaité E09 : 2.0 (poids : 1/shift)
* Jour Employé/Shift E10 S01 : 1.0
* Jour Employé/Shift E10 S04 : 1.0
* Jour Employé/Shift E10 S09 : 1.0
* Jour Employé/Shift E10 S21 : 1.0
* Employé qui fait moins de shifts que souhaité E11 : 1.0
* Semaine trop chargée Employé E13 : 1.0
* Jour Employé/Shift E17 S09 : 1.0
* Jour Employé/Shift E17 S12 : 1.0
* Semaine trop chargée Employé E18 : 1.0
  
3 (type) + 9 (jours) + 4 (shifts manquants) + 2 (semaines chargées) = 18

Si on somme tout, on a bien un total de pénalités qui vaut 18 et c'est ce que le solveur retrouve. 
